# Project pipeline

   ## Schools
mtl_schools.csv -> this notebook -> schools_json_final.geojson -> site -> my map

## Recreational sites
project_recreationsites.geojson -> this notebook -> recreation_filtered.geojson -> site -> my map
!Important to note! recreationsites was already a geojson file, it only needed to be fintered



## Part 1. Load Schools and filter the columns

This cell imports my schools and recreation files, it reads it and keeps on the relevant columns. it also creates a point with its coordinates for schools. it then also reads in the geogjson for recreational facilities, which it then filters to only keep features relevant for preschool and elementary level students. 

In [1]:
import json
import pandas as pd
import geopandas as gpd

DATA_DIR = "../data/"

# ---------- Schools ----------
keep_columns = ["COORD_X_LL84_IMM", "COORD_Y_LL84_IMM", "ORDRE_ENS", "SITE_WEB_ORGNS",
                "NOM_IMM", "CD_POSTL_GDUNO_ORGNS", "NOM_MUNCP_GDUNO_ORGNS"]
schools = pd.read_csv(DATA_DIR + "mtl_schools.csv", usecols=keep_columns)
schools = schools.dropna(subset=["COORD_X_LL84_IMM", "COORD_Y_LL84_IMM"])   # a point needs both coordinates

# Island of Montréal only: postal codes H1-H5, H8, H9 (H7 = Laval)
fsa = schools["CD_POSTL_GDUNO_ORGNS"].str.strip().str.upper()
schools = schools[fsa.str.match(r"H[1-589]", na=False)]

# Types of school to keep
wanted_orders = ["Primaire", "Préscolaire - Primaire", "Préscolaire - Primaire - Secondaire",
                 "Primaire - Secondaire", "Préscolaire"]
schools = schools[schools["ORDRE_ENS"].isin(wanted_orders)]

gdf_school = gpd.GeoDataFrame(
    schools,
    geometry=gpd.points_from_xy(schools["COORD_X_LL84_IMM"], schools["COORD_Y_LL84_IMM"]),  # lon, lat
    crs="EPSG:4326",
)

# Remove duplicate schools and give each school a number (school_id)
gdf_school = gdf_school.drop_duplicates(subset=["NOM_IMM", "COORD_X_LL84_IMM", "COORD_Y_LL84_IMM"])
gdf_school = gdf_school.reset_index(drop=True)
gdf_school["school_id"] = gdf_school.index

print("schools:", len(gdf_school), gdf_school.total_bounds)
gdf_school.to_file(DATA_DIR + "schools_json_final.geojson", driver="GeoJSON")

# ---------- Recreation sites ----------
rec = gpd.read_file(DATA_DIR + "project_recreationsites.geojson")
rec = rec[["ID", "NOM", "TYPE", "ARROND", "geometry"]]
rec = rec[rec["TYPE"].isin(["Sportif", "Récréatif", "Aire de jeu - enfant"])]

# Give each recreation site a number (rec_id)
rec = rec.reset_index(drop=True)
rec["rec_id"] = rec.index

# ---------- Seasons and age groups, derived from NOM ----------
ALL_SEASONS = ["winter", "spring", "summer", "fall"]
WARM        = ["spring", "summer", "fall"]

WINTER_ONLY = {"Patinoire avec bandes", "Patinoire sans bandes", "Patinoire réfrigérée"}
SUMMER_ONLY = {"Jeux d'eau", "Pataugeoire", "Piscine extérieure"}
ALL_YEAR    = {"Patinoire 4 saisons", "Patinoire réfrigérée / Basketball",
               "Multisport", "Bassin olympique", "Mur d'escalade", "Hébertisme urbain"}

def seasons_list(nom):
    nom = " ".join(nom.split())
    if nom in WINTER_ONLY: return ["winter"]
    if nom in SUMMER_ONLY: return ["summer"]
    if nom in ALL_YEAR or nom.startswith("Aire de jeu"): return ALL_SEASONS
    return WARM

def season_note(s):
    if s == ["winter"]:  return "Winter only"
    if s == ["summer"]:  return "Summer only"
    if s == ALL_SEASONS: return "Available all year"
    return "Spring to fall (usually not available in winter)"

def age_levels(nom):
    nom = " ".join(nom.split())
    if nom == "Aire de jeu 18 mois à 5 ans": return "Préscolaire"
    if nom == "Aire de jeu 5 à 12 ans":      return "Primaire"
    if nom == "Aire de jeu combiné":         return "Préscolaire|Primaire"
    return "all"

rec["seasons"] = rec["NOM"].apply(lambda n: "|".join(seasons_list(n)))
rec["season_note"] = rec["NOM"].apply(lambda n: season_note(seasons_list(n)))
rec["age_levels"] = rec["NOM"].apply(age_levels)

# ---------- Indoor or outdoor, derived from NOM ----------
# Names listed here count as indoor; everything else is outdoor. Edit this list as you see fit.
INDOOR = {"Bassin olympique", "Mur d'escalade"}

rec["setting"] = rec["NOM"].apply(lambda n: "Indoor" if " ".join(n.split()) in INDOOR else "Outdoor")
print(rec.groupby("setting")["NOM"].unique())

print("recreation:", len(rec))
rec.to_file(DATA_DIR + "recreation_filtered.geojson", driver="GeoJSON")

schools: 331 [-73.880687  45.419518 -73.479247  45.700858]
setting
Indoor                    [Bassin olympique, Mur d'escalade]
Outdoor    [Aire de jeu 18 mois à 5 ans, Soccer à 11, Soc...
Name: NOM, dtype: object
recreation: 2651


In [3]:
%pip install osmnx scipy scikit-learn #library which includes real walking distances in openstreetmap
import os
import osmnx as ox
import networkx as nx
from shapely.geometry import LineString, box

MAX_M = 500   # same as the slider maximum (a walk is never shorter than a straight line)
GRAPH_FILE = DATA_DIR + "walk_network.graphml"

# 1. Walking network from OpenStreetMap 
if os.path.exists(GRAPH_FILE):
    G_ll = ox.load_graphml(GRAPH_FILE)
else:
    minx, miny, maxx, maxy = gdf_school.total_bounds
    area = box(minx - 0.01, miny - 0.01, maxx + 0.01, maxy + 0.01)
    G_ll = ox.graph_from_polygon(area, network_type="walk")
    ox.save_graphml(G_ll, GRAPH_FILE)

G = ox.project_graph(G_ll)          # metres instead of degrees
crs_m = G.graph["crs"]

# 2. Schools and sites in the same metric system (meters)
sch = gdf_school.to_crs(crs_m)
rc = rec.to_crs(crs_m)

# 3. Nearest street node for each, and how far each one sits from it
sch_nodes, sch_snap = ox.distance.nearest_nodes(G, sch.geometry.x.values, sch.geometry.y.values, return_dist=True)
rc_nodes, rc_snap = ox.distance.nearest_nodes(G, rc.geometry.x.values, rc.geometry.y.values, return_dist=True)

# 4. Candidate pairs: sites within MAX_M in a straight line of each school (pairing the school to the nearest recreational sites)
buf = sch[["school_id"]].copy()
buf["geometry"] = sch.buffer(MAX_M)
buf = gpd.GeoDataFrame(buf, geometry="geometry", crs=crs_m)
pairs = gpd.sjoin(buf, rc[["rec_id", "geometry"]], how="inner", predicate="intersects")
by_school = pairs.groupby("school_id")["rec_id"].apply(list)

# 5. One street search per school, then read off the route to each candidate site
rows = []
for sid, rec_ids in by_school.items():
    dist, path = nx.single_source_dijkstra(G, sch_nodes[sid], cutoff=MAX_M, weight="length")
    school_pt = gdf_school.geometry.iloc[sid]
    for rid in rec_ids:
        target = rc_nodes[rid]
        if target not in dist:
            continue                                   # no walkable route within MAX_M
        walk_m = dist[target] + sch_snap[sid] + rc_snap[rid]
        if walk_m > MAX_M:
            continue
        site_pt = rec.geometry.iloc[rid]
        pts = ([(school_pt.x, school_pt.y)]
               + [(G_ll.nodes[n]["x"], G_ll.nodes[n]["y"]) for n in path[target]]
               + [(site_pt.x, site_pt.y)])
        rows.append({"school_id": int(sid), "rec_id": int(rid),
                     "walk_m": round(walk_m), "geometry": LineString(pts)})

routes = gpd.GeoDataFrame(rows, crs="EPSG:4326")
routes["geometry"] = routes.geometry.simplify(0.00002)    # about 2 m, keeps the file small
routes.to_file(DATA_DIR + "walking_routes.geojson", driver="GeoJSON")

print(len(routes), "routes;", routes["school_id"].nunique(), "of", len(gdf_school), "schools have at least one site in reach")
print(round(os.path.getsize(DATA_DIR + "walking_routes.geojson") / 1e6, 1), "MB")



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#library': Expected package name at the start of dependency specifier
    #library
    ^


Note: you may need to restart the kernel to use updated packages.
1293 routes; 270 of 331 schools have at least one site in reach
0.5 MB


##  Check the files

this check file will varify if all the data i need is included and if there is any missing data within my file.

In [2]:


problems = []

# Schools file
with open(DATA_DIR + "schools_json_final.geojson", encoding="utf-8") as f:
    back = json.load(f)

feats = back.get("features", [])
if back.get("type") != "FeatureCollection":
    problems.append("The schools file is not a FeatureCollection.")
if len(feats) != len(gdf_school):
    problems.append(f"The schools file has {len(feats)} features but the clean table has {len(gdf_school)} rows.")
lons = [ft["geometry"]["coordinates"][0] for ft in feats]
lats = [ft["geometry"]["coordinates"][1] for ft in feats]
if feats and not all(-74.1 < x < -73.4 for x in lons):
    problems.append("Some school longitudes are outside Montréal. Are longitude and latitude swapped?")
if feats and not all(45.3 < y < 45.8 for y in lats):                                                    #checnking if long and lat are changed?
    problems.append("Some school latitudes are outside Montréal. Are longitude and latitude swapped?")
if feats and not all({"NOM_IMM", "ORDRE_ENS", "CD_POSTL_GDUNO_ORGNS"} <= set(ft["properties"]) for ft in feats):
    problems.append("Some schools are missing NOM_IMM, ORDRE_ENS or CD_POSTL_GDUNO_ORGNS in their properties.")  # checnking for missing values
if feats and not all(ft["properties"]["ORDRE_ENS"] in wanted_orders for ft in feats):
    problems.append("Some schools have a school type you meant to remove.")  #checking the school types

# Recreation file
with open(DATA_DIR + "recreation_filtered.geojson", encoding="utf-8") as f:
    back_r = json.load(f)

feats_r = back_r.get("features", [])
if back_r.get("type") != "FeatureCollection":
    problems.append("The recreation file is not a FeatureCollection.")
if len(feats_r) != len(rec):
    problems.append(f"The recreation file has {len(feats_r)} features but the clean table has {len(rec)} rows.")
minx, miny, maxx, maxy = gpd.read_file(DATA_DIR + "recreation_filtered.geojson").total_bounds
if not (-74.1 < minx and maxx < -73.4 and 45.3 < miny and maxy < 45.8):  #same as schools, checking the coordinates
    problems.append("Some recreation sites are outside Montréal. Wrong coordinate system, or swapped coordinates?")
if feats_r and not all({"ID", "NOM", "TYPE", "ARROND"} <= set(ft["properties"]) for ft in feats_r):
    problems.append("Some recreation sites are missing ID, NOM, TYPE or ARROND in their properties.")  #checking for missing values
if feats_r and not all(ft["properties"]["TYPE"] in ["Sportif", "Récréatif", "Aire de jeu - enfant"] for ft in feats_r):
    problems.append("Some recreation sites have a TYPE you meant to remove.")  # checking the file to have the right data

print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))


All checks passed
